In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

import numpy as np
import pandas as pd

import lime
import lime.lime_tabular
import shap

from copy import deepcopy
import json

from rule_of_thumb import RuleOfThumb

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)


In [2]:
import os
import random
import time
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [5]:
from lime.lime_tabular import LimeTabularExplainer

In [66]:
os.makedirs('./plots', exist_ok=True)

In [6]:

fn_training_set = 'data/output/datasets/training_set.csv.xz'
fn_pairwise = 'data/output/datasets/pairwise_training_set.csv.gz'

In [7]:

random_seed = 303
random.seed(random_seed)

In [8]:
training_data = pd.read_csv(fn_pairwise, compression='gzip')

In [9]:
# features
cols = ['stars_delta', 'reviews_delta', 'is_amazon',
        'is_shipped_by_amazon', 'is_sold_by_amazon',
        'is_top_clicked', 'random_noise']
# target
target = 'placed_higher'

In [10]:
all_x = training_data[cols]
all_y = training_data[target].astype(int)

In [11]:
xtrain, xtest, ytrain, ytest = train_test_split(all_x, all_y, test_size=0.2, random_state=random_seed)

In [12]:
scaler = StandardScaler()
xtrain_np = scaler.fit_transform(xtrain)
xtrain = pd.DataFrame(xtrain_np, columns=xtrain.columns)
xtest_np = scaler.transform(xtest)
xtest = pd.DataFrame(xtest_np, columns=xtest.columns)

In [13]:
X_train, X_test, y_train, y_test = xtrain, xtest, ytrain, ytest

In [14]:
# Assuming X_train, y_train, X_test, y_test are the training and test data

# Train Random Forest Classifier
rf_classifier = RandomForestClassifier(max_depth=3, max_features=6, n_estimators=500, random_state=random_seed)
rf_classifier.fit(X_train, y_train)
rf_predictions = rf_classifier.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)


rf_def = RandomForestClassifier(random_state=random_seed)
rf_def.fit(X_train, y_train)
rf_def_predictions = rf_def.predict(X_test)
rf_def_accuracy = accuracy_score(y_test, rf_def_predictions)

# Train Logistic Regression
logistic_classifier = LogisticRegression(max_iter=1000, random_state=random_seed)
logistic_classifier.fit(X_train, y_train)
logistic_predictions = logistic_classifier.predict(X_test)
logistic_accuracy = accuracy_score(y_test, logistic_predictions)

# Train Logistic Regression with L1 regularization
logistic_l1_classifier = LogisticRegression(penalty='l1', solver='liblinear', C=0.01, max_iter=1000, random_state=random_seed)
logistic_l1_classifier.fit(X_train, y_train)
logistic_l1_predictions = logistic_l1_classifier.predict(X_test)
logistic_l1_accuracy = accuracy_score(y_test, logistic_l1_predictions)

# Train Logistic Regression with L2 regularization
logistic_l2_classifier = LogisticRegression(penalty='l2', max_iter=1000, C=0.01, random_state=random_seed)
logistic_l2_classifier.fit(X_train, y_train)
logistic_l2_predictions = logistic_l2_classifier.predict(X_test)
logistic_l2_accuracy = accuracy_score(y_test, logistic_l2_predictions)

# Print accuracies
print("Markup Random Forest Accuracy:", rf_accuracy)
print("Default Random Forest Accuracy:", rf_def_accuracy)
print("Logistic Regression Accuracy:", logistic_accuracy)
print("Logistic Regression (L1) Accuracy:", logistic_l1_accuracy)
print("Logistic Regression (L2) Accuracy:", logistic_l2_accuracy)


Markup Random Forest Accuracy: 0.6855123674911661
Default Random Forest Accuracy: 0.6890459363957597
Logistic Regression Accuracy: 0.7102473498233216
Logistic Regression (L1) Accuracy: 0.7243816254416962
Logistic Regression (L2) Accuracy: 0.6925795053003534


In [17]:
rf_importances = rf_classifier.feature_importances_ / np.sum(rf_classifier.feature_importances_)
rf_def_importances = rf_def.feature_importances_ / np.sum(rf_def.feature_importances_)
logistic_importances = np.abs(logistic_classifier.coef_[0]) / np.sum(np.abs(logistic_classifier.coef_[0]))
logistic_l1_importances = np.abs(logistic_l1_classifier.coef_[0]) / np.sum(np.abs(logistic_l1_classifier.coef_[0]))
logistic_l2_importances = np.abs(logistic_l2_classifier.coef_[0]) / np.sum(np.abs(logistic_l2_classifier.coef_[0]))

# Create a DataFrame to store feature importances
feature_importance_df = pd.DataFrame({
    'Random Forest': rf_importances,
    'Logistic Regression': logistic_importances,
    'Logistic Regression (L1)': logistic_l1_importances,
    'Logistic Regression (L2)': logistic_l2_importances
}, index=X_train.columns)

In [18]:
feature_importances = {
    'Markup Random Forest': rf_importances,
    'Default Random Forest': rf_def_importances,
    'Logistic Regression': logistic_importances,
    'Logistic Regression (L1)': logistic_l1_importances,
    'Logistic Regression (L2)': logistic_l2_importances,
}

feature_importance_df = pd.DataFrame(feature_importances, index=X_train.columns)



In [21]:
xx = xtrain.values
yy = ytrain.values
rot_start = time.time()
rot1 = RuleOfThumb(yy, xx)
rot_middle = time.time()
xx = xtest.values
rot_exps1 = rot1.get_explanation(xx)
# some_exps_unused = rot1.get_explanation(xtest.values)
rot_end = time.time()
# rot_results1 = []
# for exp in rot_exps1:
#     _temp = [(_feat, _exp) for _feat, _exp in zip(features, exp)]
#     rot_results1.append(experiment_summary([_temp], features))


In [22]:

rot_predictions = rot1._explainer_model.predict(torch.from_numpy(xtest.values).to(torch.float)).detach().numpy()

In [23]:
accuracy_score(rot_predictions, ytest.values)

0.696113074204947

In [26]:
rot_imps = pd.DataFrame(rot1.get_explanation(xtest.values), columns=cols)

In [27]:
raw_rot_globals = abs(rot_imps).mean()


In [28]:
raw_rot_globals /= abs(raw_rot_globals).sum()

In [29]:
print(f"{raw_rot_globals=}")

raw_rot_globals=stars_delta             0.074781
reviews_delta           0.007212
is_amazon               0.620805
is_shipped_by_amazon    0.001870
is_sold_by_amazon       0.278919
is_top_clicked          0.001734
random_noise            0.014678
dtype: float64


In [30]:
lime_markup_start = time.time()
lime_explainer = LimeTabularExplainer(
    training_data=xtrain.values,
    feature_names=xtrain.columns.tolist(),
)

# Collect explanations in array format [n_samples, n_features]
lime_explanations = np.zeros((len(xtest), xtrain.shape[1]))

lime_markup_middle = time.time()

for i in range(len(xtest)):
    exp = lime_explainer.explain_instance(
        data_row=xtest.iloc[i].values,
        predict_fn=lambda x: rf_classifier.predict_proba(
            pd.DataFrame(x, columns=xtrain.columns)
        )
    )
    # Convert explanation for class 1 into a dict {feature_index: weight}
    exp_map = dict(exp.as_map()[1])  # class index 1
    for feat_idx, weight in exp_map.items():
        lime_explanations[i, feat_idx] = weight


lime_markup_imps = pd.DataFrame(lime_explanations, columns=xtrain.columns)
lime_markup_end = time.time()

In [31]:
lime_def_start = time.time()
lime_explainer = LimeTabularExplainer(
    training_data=xtrain.values,
    feature_names=xtrain.columns.tolist(),
)

# Collect explanations in array format [n_samples, n_features]
lime_explanations = np.zeros((len(xtest), xtrain.shape[1]))

lime_def_middle = time.time()

for i in range(len(xtest)):
    exp = lime_explainer.explain_instance(
        data_row=xtest.iloc[i].values,
        predict_fn=lambda x: rf_def.predict_proba(
            pd.DataFrame(x, columns=xtrain.columns)
        )
    )
    # Convert explanation for class 1 into a dict {feature_index: weight}
    exp_map = dict(exp.as_map()[1])  # class index 1
    for feat_idx, weight in exp_map.items():
        lime_explanations[i, feat_idx] = weight


lime_def_imps = pd.DataFrame(lime_explanations, columns=xtrain.columns)
lime_def_end = time.time()

In [32]:
lime_lr_start = time.time()
lime_explainer = LimeTabularExplainer(
    training_data=xtrain.values,
    feature_names=xtrain.columns.tolist(),
)

# Collect explanations in array format [n_samples, n_features]
lime_explanations = np.zeros((len(xtest), xtrain.shape[1]))

lime_lr_middle = time.time()

for i in range(len(xtest)):
    exp = lime_explainer.explain_instance(
        data_row=xtest.iloc[i].values,
        predict_fn=lambda x: logistic_classifier.predict_proba(
            pd.DataFrame(x, columns=xtrain.columns)
        )
    )
    # Convert explanation for class 1 into a dict {feature_index: weight}
    exp_map = dict(exp.as_map()[1])  # class index 1
    for feat_idx, weight in exp_map.items():
        lime_explanations[i, feat_idx] = weight


lime_lr_imps = pd.DataFrame(lime_explanations, columns=xtrain.columns)
lime_lr_end = time.time()

In [33]:
lime_l1_start = time.time()
lime_explainer = LimeTabularExplainer(
    training_data=xtrain.values,
    feature_names=xtrain.columns.tolist(),
)

# Collect explanations in array format [n_samples, n_features]
lime_explanations = np.zeros((len(xtest), xtrain.shape[1]))

lime_l1_middle = time.time()

for i in range(len(xtest)):
    exp = lime_explainer.explain_instance(
        data_row=xtest.iloc[i].values,
        predict_fn=lambda x: logistic_l1_classifier.predict_proba(
            pd.DataFrame(x, columns=xtrain.columns)
        )
    )
    # Convert explanation for class 1 into a dict {feature_index: weight}
    exp_map = dict(exp.as_map()[1])  # class index 1
    for feat_idx, weight in exp_map.items():
        lime_explanations[i, feat_idx] = weight


lime_l1_imps = pd.DataFrame(lime_explanations, columns=xtrain.columns)
lime_l1_end = time.time()

In [34]:
lime_l2_start = time.time()
lime_explainer = LimeTabularExplainer(
    training_data=xtrain.values,
    feature_names=xtrain.columns.tolist(),
)

# Collect explanations in array format [n_samples, n_features]
lime_explanations = np.zeros((len(xtest), xtrain.shape[1]))

lime_l2_middle = time.time()

for i in range(len(xtest)):
    exp = lime_explainer.explain_instance(
        data_row=xtest.iloc[i].values,
        predict_fn=lambda x: logistic_l2_classifier.predict_proba(
            pd.DataFrame(x, columns=xtrain.columns)
        )
    )
    # Convert explanation for class 1 into a dict {feature_index: weight}
    exp_map = dict(exp.as_map()[1])  # class index 1
    for feat_idx, weight in exp_map.items():
        lime_explanations[i, feat_idx] = weight

lime_l2_imps = pd.DataFrame(lime_explanations, columns=xtrain.columns)
lime_l2_end = time.time()


# load shap stuff into memory to reduce first init time

In [35]:
# load shap stuff into memory to reduce first init time
_  = shap.KernelExplainer(lambda x: rot1._explainer_model.predict(torch.from_numpy(x).to(torch.float)).detach().numpy(), xtrain)

Using 1132 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


In [36]:
shap_markup_start = time.time()
# background_distribution = shap.kmeans(xtrain,10)
# shap_explainer = shap.KernelExplainer(rf_classifier.predict, background_distribution)
shap_explainer = shap.KernelExplainer(rf_classifier.predict, xtrain)
shap_markup_middle = time.time()
explanations = shap_explainer.shap_values(xtest)
shap_markup_end = time.time()

Using 1132 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/283 [00:00<?, ?it/s]

In [37]:

shap_markup_imps = pd.DataFrame(explanations, columns=cols)

In [39]:
shap_def_start = time.time()
# background_distribution = shap.kmeans(xtrain,10)
# shap_def_explainer = shap.KernelExplainer(rf_def.predict, background_distribution)
shap_def_explainer = shap.KernelExplainer(rf_def.predict, xtrain)
shap_def_middle = time.time()
explanations_def = shap_def_explainer.shap_values(xtest)
shap_def_end = time.time()

Using 1132 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/283 [00:00<?, ?it/s]

In [40]:
shap_def_imps = pd.DataFrame(explanations_def, columns=cols)

In [42]:
shap_lr_start = time.time()
shap_lr_explainer = shap.KernelExplainer(logistic_classifier.predict, xtrain)
shap_lr_middle = time.time()
explanations_lr = shap_lr_explainer.shap_values(xtest)
shap_lr_end = time.time()

Using 1132 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/283 [00:00<?, ?it/s]

In [43]:
shap_lr_imps = pd.DataFrame(explanations_lr, columns=cols)

In [45]:
shap_l1_start = time.time()
shap_l1_explainer = shap.KernelExplainer(logistic_l1_classifier.predict, xtrain)
shap_l1_middle = time.time()
explanations_l1 = shap_l1_explainer.shap_values(xtest)
shap_l1_end = time.time()
shap_l1_imps = pd.DataFrame(explanations_l1, columns=cols)
# plot_rot(shap_l1_imps, 'LogReg (L1) SHAP importances')

Using 1132 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/283 [00:00<?, ?it/s]

In [46]:
shap_l2_start = time.time()
shap_l2_explainer = shap.KernelExplainer(logistic_l2_classifier.predict, xtrain)
shap_l2_middle = time.time()
explanations_l2 = shap_l2_explainer.shap_values(xtest)
shap_l2_end = time.time()
shap_l2_imps = pd.DataFrame(explanations_l2, columns=cols)
# plot_rot(shap_l2_imps, 'LogReg (L2) SHAP importances')

Using 1132 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/283 [00:00<?, ?it/s]

In [62]:


def plot_new_stack_two_dfs(df1, df2, title, filename=None):
    if filename is None:
        filename = f'{title}.png'

    def process_df(original_df):
        df = abs(original_df.copy())
        df['1st_highest'] = abs(original_df).apply(lambda row: row.nlargest(3).index[0], axis=1)
        df['2nd_highest'] = abs(original_df).apply(lambda row: row.nlargest(3).index[1], axis=1)
        df['3rd_highest'] = abs(original_df).apply(lambda row: row.nlargest(3).index[2], axis=1)
        df['4_highest'] = abs(original_df).apply(lambda row: row.nlargest(4).index[3], axis=1)
        df['5_highest'] = abs(original_df).apply(lambda row: row.nlargest(5).index[4], axis=1)
        df['6_highest'] = abs(original_df).apply(lambda row: row.nlargest(6).index[5], axis=1)
        df['7_highest'] = abs(original_df).apply(lambda row: row.nlargest(7).index[6], axis=1)
        _counts_df = pd.DataFrame({
            'Most Important': [sum(df[df['1st_highest'] == col][col]) for col in original_df.columns],
            'Second Most Important': [sum(df[df['2nd_highest'] == col][col]) for col in original_df.columns],
            'Third': [sum(df[df['3rd_highest'] == col][col]) for col in original_df.columns],
            'Fourth': [sum(df[df['4_highest'] == col][col]) for col in original_df.columns],
            'Fifth': [sum(df[df['5_highest'] == col][col]) for col in original_df.columns],
            'Sixth': [sum(df[df['6_highest'] == col][col]) for col in original_df.columns],
            'Least Important': [sum(df[df['7_highest'] == col][col]) for col in original_df.columns],
        }, index=original_df.columns)

        counts_df = _counts_df.copy()
        counts_df['Third or Lesser Importance'] = (
            counts_df['Third'] + counts_df['Fourth'] + counts_df['Fifth'] +
            counts_df['Sixth'] + counts_df['Least Important']
        )
        counts_df = counts_df.drop(columns=['Third', 'Fourth', 'Fifth', 'Sixth', 'Least Important'])

        counts_df_T = counts_df.T
        counts_df_T = counts_df_T.rename(columns={
            'stars_delta': 'product\nstars',
            'reviews_delta': 'product\nreviews',
            'is_amazon': 'brand is\namazon',
            'is_sold_by_amazon': 'sold by\namazon',
            'is_shipped_by_amazon': 'shipped\nby amazon',
            'is_top_clicked': 'top\nclicked',
            'random_noise': 'random\nnoise'
        })
        return counts_df_T.T

    counts_df1 = process_df(df1)
    counts_df2 = process_df(df2)
    # print(f'{counts_df1.index=}')
    plot_order = [
        'brand is\namazon',
        'product\nreviews',
        'sold by\namazon',
        'top\nclicked',
        'product\nstars',
        'shipped\nby amazon',
        'random\nnoise'
    ]
    counts_df1 = counts_df1.reindex(plot_order)
    counts_df2 = counts_df2.reindex(plot_order)
    # print(f'{counts_df2.index=}')
    # print('======================')


    # plotting
    fig, ax1 = plt.subplots(figsize=(12,10))
    ax2 = ax1.twinx()  # second y-axis

    x = range(len(counts_df1))
    # barcolors = ['tab:cyan', 'tab:pink', 'tab:olive']
    barcolors = ['darkgreen', 'fuchsia', 'gold']
    width = 0.16
    gap = 0.14  # extra spacing between df1 and df2 bars

    # Plot df2 (hatched, left, ax2)
    bottom = [0] * len(counts_df2)
    for i, col in enumerate(counts_df2.columns):
        ax2.bar(
            [p - width/2 - gap/2 - 0.03 for p in x], counts_df2[col], bottom=bottom,
            width=width, facecolor='none', edgecolor=barcolors[i],
            hatch='//', linewidth=5,
            label=f"{col} (df2)"
        )
        bottom = [b + v + 0 for b, v in zip(bottom, counts_df2[col])]

    # Plot df1 (solid, right, ax1)
    bottom = [0] * len(counts_df1)
    for i, col in enumerate(counts_df1.columns):
        ax1.bar(
            [p + width/2 + gap/2 + 0.03 for p in x], counts_df1[col], bottom=bottom,
            width=width+0.06, facecolor=barcolors[i], edgecolor='white',
            linewidth=5, label=f"{col} (df1)"
        )
        bottom = [b + v + 0 for b, v in zip(bottom, counts_df1[col])]

    # ticks & labels
    ax1.set_xticks(x)
    ax1.set_xticklabels(counts_df1.index, rotation=90, fontsize=30)

    ax1.set_xlabel('Scraped Feature', fontsize=35)
    ax1.set_ylabel('$\Sigma$ $abs($ SHAP importance $)$', fontsize=35)
    ax2.set_ylabel('$\Sigma$ $abs($ LIME importance $)$', fontsize=35)

    # merge legends (6 entries: 3 solid + 3 hatched)
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    # chand = [plt.plot([],marker="", ls="")[0]] + handles1 + [plt.plot([],marker="", ls="")[0]] + handles2
    # clab = ["SHAP"] + [' ',' ',' '] + ["LIME"] + labels2
    clab = ["      Most\n  Important", '  SHAP  ', '  LIME  '] + [" Second Most \n   Important", '  SHAP', '  LIME'] + [" Third or Lesser\n    Importance", '  SHAP', '  LIME']
    chand = [plt.plot([],marker="", ls="")[0]] + [handles1[0]] + [handles2[0]] + [plt.plot([],marker="", ls="")[0]] + [handles1[1]] + [handles2[1]] + [plt.plot([],marker="", ls="")[0]] + [handles1[2]] + [handles2[2]]
    
    leg = ax1.legend(chand, clab,
               title='Relative Importance per Datapoint',
               fontsize=28, title_fontproperties={'weight': 'bold', 'size': 29}, ncols=3, loc='upper right', columnspacing=0.5, handletextpad=0, borderaxespad=0, borderpad=0.3, labelspacing=0.2)

    for vpack in leg._legend_handle_box.get_children():
        for hpack in vpack.get_children()[:1]:
            hpack.get_children()[0].set_width(0)
    
    # expand ax2 ylim by +10 on the upper side
    multiplier = 1.4
    ymin, ymax = ax2.get_ylim()
    ax2.set_ylim(ymin, ymax * multiplier)
    ymin, ymax = ax1.get_ylim()
    ax1.set_ylim(ymin, ymax * multiplier)


    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()


In [67]:
# plot_new_stack(rot_imps, 'RoT Importances', 'plots/rot_bar.pdf')
plot_new_stack_two_dfs(shap_markup_imps, lime_markup_imps, 'LIME+SHAP (Markup RF) Importances', 'plots/sl-markup_bar.pdf')
plot_new_stack_two_dfs(shap_def_imps, lime_def_imps, 'LIME+SHAP (Default RF) Importances', 'plots/sl-rf_bar.pdf')
plot_new_stack_two_dfs(shap_lr_imps, lime_lr_imps, 'LIME+SHAP (LR) Importances', 'plots/sl-lr-bar.pdf')
plot_new_stack_two_dfs(shap_l1_imps, lime_l1_imps, 'LIME+SHAP (LR-l1) Importances', 'plots/sl-l1-bar.pdf')
plot_new_stack_two_dfs(shap_l2_imps, lime_l2_imps, 'LIME+SHAP (LR-l2) Importances', 'plots/sl-l2-bar.pdf')


In [68]:


def plot_new_stack(original_df, title, filename=None):
    if filename is None:
        filename = f'{title}.png'
    plt.figure().clf()
    df = abs(original_df.copy())
    df['1st_highest'] = abs(original_df).apply(lambda row: row.nlargest(3).index[0], axis=1)
    df['2nd_highest'] = abs(original_df).apply(lambda row: row.nlargest(3).index[1], axis=1)
    df['3rd_highest'] = abs(original_df).apply(lambda row: row.nlargest(3).index[2], axis=1)
    df['4_highest'] = abs(original_df).apply(lambda row: row.nlargest(4).index[3], axis=1)
    df['5_highest'] = abs(original_df).apply(lambda row: row.nlargest(5).index[4], axis=1)
    df['6_highest'] = abs(original_df).apply(lambda row: row.nlargest(6).index[5], axis=1)
    df['7_highest'] = abs(original_df).apply(lambda row: row.nlargest(7).index[6], axis=1)
    _counts_df = pd.DataFrame({
        'Most Important': [sum(df[df['1st_highest'] == col][col]) for col in original_df.columns],
        'Second Most Important': [sum(df[df['2nd_highest'] == col][col]) for col in original_df.columns],
        'Third': [sum(df[df['3rd_highest'] == col][col]) for col in original_df.columns],
        'Fourth': [sum(df[df['4_highest'] == col][col]) for col in original_df.columns],
        'Fifth': [sum(df[df['5_highest'] == col][col]) for col in original_df.columns],
        'Sixth': [sum(df[df['6_highest'] == col][col]) for col in original_df.columns],
        'Least Important': [sum(df[df['7_highest'] == col][col]) for col in original_df.columns],
    }, index=original_df.columns)

    counts_df = _counts_df.copy()
    counts_df['Third or Lesser Importance'] = counts_df['Third'] + counts_df['Fourth'] + counts_df['Fifth'] + counts_df['Sixth'] + counts_df['Least Important']
    counts_df = counts_df.drop(columns=['Third', 'Fourth', 'Fifth', 'Sixth', 'Least Important'])

    counts_df_T = counts_df.T
    # counts_df_T = counts_df_T.rename(columns={'stars_delta': 'stars\ndelta', 'reviews_delta': 'reviews\ndelta', 'is_amazon': 'is\namazon', 'is_sold_by_amazon': 'is sold by\namazon', 'is_shipped_by_amazon': 'is shipped\nby amazon', 'is_top_clicked': 'is top\nclicked', 'random_noise': 'random\nnoise'})
    counts_df_T = counts_df_T.rename(columns={'stars_delta': 'product\nstars', 'reviews_delta': 'product\nreviews', 'is_amazon': 'brand is\namazon', 'is_sold_by_amazon': 'sold by\namazon', 'is_shipped_by_amazon': 'shipped\nby amazon', 'is_top_clicked': 'top\nclicked', 'random_noise': 'random\nnoise'})
    counts_df = counts_df_T.T
    # print(counts_df)
    plot_order = [
        'brand is\namazon',
        'product\nreviews',
        'sold by\namazon',
        'top\nclicked',
        'product\nstars',
        'shipped\nby amazon',
        'random\nnoise'
    ]
    counts_df = counts_df.reindex(plot_order)

    
    
    # plt.figure(figsize=(8,5))
    plt.figure(figsize=(12,10))
    # Prepare x positions
    x = range(len(counts_df))
    # Initialize bottom (for stacking)
    bottom = [0] * len(counts_df)
    # Plot each column as a stacked layer
    # barcolors = ['tab:cyan', 'tab:pink', 'tab:olive']
    barcolors = ['darkgreen', 'fuchsia', 'gold']
    # barhatches = ['///', 'o', '\\']
    barhatches = [None, None, None]
    for i, col in enumerate(counts_df.columns):
        plt.bar(x, counts_df[col], bottom=bottom, label=col, width=0.16 + 0.06, facecolor=barcolors[i], edgecolor='white', fill=True, hatch=barhatches[i], linewidth=5)
        bottom = [b + v for b, v in zip(bottom, counts_df[col])]
    # Labels, legend, etc.
    plt.xticks(x, counts_df.index, rotation=90, fontsize=30)
    
    # counts_df.plot(kind='bar', stacked=True, figsize=(8,6), width=0.2, edgecolor='black')#, ax=ax)
    # plt.xticks(rotation=0, fontsize=13)

    if filename == 'plots/l1-bar.pdf':
        plt.ylim((0,109))
    
    plt.xlabel('Feature Name', fontsize=35)
    # print(f'{filename=}')
    # print(f'{title=}')
    if filename == 'plots/rot_bar.pdf':
        # print('RoT')
        plt.ylabel('$\Sigma$ $abs($ RoT importance $)$', fontsize=35)
    elif 'SHAP' in title:
        # print('SHAP')
        plt.ylabel('$\Sigma$ $abs($ SHAP importance $)$', fontsize=35)
    else:
        # print('LIME')
        plt.ylabel('$\Sigma$ $abs($ LIME importance $)$', fontsize=35)
    multiplier = 1.1
    ymin, ymax = plt.ylim()
    plt.ylim(ymin, ymax * multiplier)
    # plt.ylim(0,160)
    # plt.title(title)
    plt.legend(title='Relative Importance per Datapoint',
               fontsize=28, title_fontproperties={'weight': 'bold', 'size': 29})#, loc='upper right', columnspacing=0.5, handletextpad=0, borderaxespad=0, borderpad=0.3, labelspacing=0.2)
    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()
    # plt.show()



def vplot(df, title, filename=None):
    if filename is None:
        filename = f'{title}.png'
    plt.figure().clf()
    # sns.boxplot(data = pd.DataFrame(rot1.get_explanation(xtest.values), columns=cols), whis=(0,100))
    vpdf = df # df[['stars_delta', 'reviews_delta', 'is_amazon', 'is_shipped_by_amazon', 'is_sold_by_amazon']]
    vpdf = vpdf.rename(columns={'stars_delta': 'product\nstars', 'reviews_delta': 'product\nreviews', 'is_amazon': 'brand is\namazon', 'is_sold_by_amazon': 'sold by\namazon', 'is_shipped_by_amazon': 'shipped\nby amazon', 'is_top_clicked': 'top\nclicked', 'random_noise': 'random\nnoise'})
    # print(vpdf)
    # plt.figure(figsize=(7, 4))
    plt.figure(figsize=(8,6))
    sns.violinplot(data = vpdf)
    plt.xlabel('Feature Name', fontsize=14)
    plt.ylabel('Importance', fontsize=14)
    plt.xticks(rotation=0, fontsize=13)
    # plt.ylim(-1.75,1.75)
    # plt.title(title)
    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()
    

In [69]:
vplot(rot_imps / 2, 'RoT Importance Distribution', 'plots/rot_violin.pdf')
vplot(shap_markup_imps, 'SHAP (Markup RF) Importance Distribution', 'plots/markup_violin.pdf')
vplot(shap_def_imps, 'SHAP (Default RF) Importance Distribution', 'plots/rf_violin.pdf')
vplot(shap_lr_imps, 'SHAP (LR) Importance Distribution', 'plots/lr_violin.pdf')
vplot(shap_l1_imps, 'SHAP (LR-l1) Importance Distribution', 'plots/l1-violin.pdf')
vplot(shap_l2_imps, 'SHAP (LR-l2) Importance Distribution', 'plots/l2-violin.pdf')

# vplot(rot_imps / 2, 'RoT Importance Distribution', 'plots/rot_violin.pdf')
vplot(lime_markup_imps, 'LIME (Markup RF) Importance Distribution', 'plots/l_markup_violin.pdf')
vplot(lime_def_imps, 'LIME (Default RF) Importance Distribution', 'plots/l_rf_violin.pdf')
vplot(lime_lr_imps, 'LIME (LR) Importance Distribution', 'plots/l_lr_violin.pdf')
vplot(lime_l1_imps, 'LIME (LR-l1) Importance Distribution', 'plots/l_l1-violin.pdf')
vplot(lime_l2_imps, 'LIME (LR-l2) Importance Distribution', 'plots/l_l2-violin.pdf')

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [70]:
plot_new_stack(rot_imps, 'RoT Importances', 'plots/rot_bar.pdf')
plot_new_stack(shap_markup_imps, 'SHAP (Markup RF) Importances', 'plots/markup_bar.pdf')
plot_new_stack(shap_def_imps, 'SHAP (Default RF) Importances', 'plots/rf_bar.pdf')
plot_new_stack(shap_lr_imps, 'SHAP (LR) Importances', 'plots/lr_bar.pdf')
plot_new_stack(shap_l1_imps, 'SHAP (LR-l1) Importances', 'plots/l1-bar.pdf')
plot_new_stack(shap_l2_imps, 'SHAP (LR-l2) Importances', 'plots/l2-bar.pdf')

plot_new_stack(lime_markup_imps, 'LIME (Markup RF) Importances', 'plots/l_markup_bar.pdf')
plot_new_stack(lime_def_imps, 'LIME (Default RF) Importances', 'plots/l_rf_bar.pdf')
plot_new_stack(lime_lr_imps, 'LIME (LR) Importances', 'plots/l_lr_bar.pdf')
plot_new_stack(lime_l1_imps, 'LIME (LR-l1) Importances', 'plots/l_l1-bar.pdf')
plot_new_stack(lime_l2_imps, 'LIME (LR-l2) Importances', 'plots/l_l2-bar.pdf')

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [73]:
print(f'Time taken for RoT [init]: {rot_middle - rot_start}')

print(f'Time taken for Markup SHAP [init]: {shap_markup_middle - shap_markup_start}')
print(f'Time taken for Default SHAP [init]: {shap_def_middle - shap_def_start}')
print(f'Time taken for LR SHAP [init]: {shap_lr_middle - shap_lr_start}')
print(f'Time taken for LR-l1 SHAP [init]: {shap_l1_middle - shap_l1_start}')
print(f'Time taken for LR-l2 SHAP [init]: {shap_l2_middle - shap_l2_start}')

print(f'Time taken for Markup LIME [init]: {lime_markup_middle - lime_markup_start}')
print(f'Time taken for Default LIME [init]: {lime_def_middle - lime_def_start}')
print(f'Time taken for LR LIME [init]: {lime_lr_middle - lime_lr_start}')
print(f'Time taken for LR-l1 LIME [init]: {lime_l1_middle - lime_l1_start}')
print(f'Time taken for LR-l2 LIME [init]: {lime_l2_middle - lime_l2_start}')

Time taken for RoT [init]: 0.6732428073883057
Time taken for Markup SHAP [init]: 0.05002903938293457
Time taken for Default SHAP [init]: 0.014747142791748047
Time taken for LR SHAP [init]: 0.0010879039764404297
Time taken for LR-l1 SHAP [init]: 0.018313884735107422
Time taken for LR-l2 SHAP [init]: 0.004772186279296875
Time taken for Markup LIME [init]: 0.003010988235473633
Time taken for Default LIME [init]: 0.0031609535217285156
Time taken for LR LIME [init]: 0.005499124526977539
Time taken for LR-l1 LIME [init]: 0.005614757537841797
Time taken for LR-l2 LIME [init]: 0.0035169124603271484


In [74]:
print(f'Time taken for RoT: {rot_end - rot_start}')

print(f'Time taken for Markup SHAP: {shap_markup_end - shap_markup_start}')
print(f'Time taken for Default SHAP: {shap_def_end - shap_def_start}')
print(f'Time taken for LR SHAP: {shap_lr_end - shap_lr_start}')
print(f'Time taken for LR-l1 SHAP: {shap_l1_end - shap_l1_start}')
print(f'Time taken for LR-l2 SHAP: {shap_l2_end - shap_l2_start}')

print(f'Time taken for Markup LIME: {lime_markup_end - lime_markup_start}')
print(f'Time taken for Default LIME: {lime_def_end - lime_def_start}')
print(f'Time taken for LR LIME: {lime_lr_end - lime_lr_start}')
print(f'Time taken for LR-l1 LIME: {lime_l1_end - lime_l1_start}')
print(f'Time taken for LR-l2 LIME: {lime_l2_end - lime_l2_start}')

Time taken for RoT: 0.6750078201293945
Time taken for Markup SHAP: 294.57070112228394
Time taken for Default SHAP: 184.17590284347534
Time taken for LR SHAP: 82.19926023483276
Time taken for LR-l1 SHAP: 81.58718276023865
Time taken for LR-l2 SHAP: 81.50589799880981
Time taken for Markup LIME: 21.24118685722351
Time taken for Default LIME: 11.15864086151123
Time taken for LR LIME: 2.158811092376709
Time taken for LR-l1 LIME: 1.875216007232666
Time taken for LR-l2 LIME: 1.8898839950561523


In [75]:
rtime = rot_end - rot_start
smtime = shap_markup_end - shap_markup_start
sdtime = shap_def_end - shap_def_start
sltime = shap_lr_end - shap_lr_start
sl1time = shap_l1_end - shap_l1_start
sl2time = shap_l2_end - shap_l2_start

print(f'Speedup over Markup SHAP: {smtime / rtime}')
print(f'Speedup over Default SHAP: {sdtime / rtime}')
print(f'Speedup LR SHAP: {sltime / rtime}')
print(f'Speedup LR-l1 SHAP: {sl1time / rtime}')
print(f'Speedup LR-l2 SHAP: {sl1time / rtime}')


lmtime = lime_markup_end - lime_markup_start
ldtime = lime_def_end - lime_def_start
lltime = lime_lr_end - lime_lr_start
ll1time = lime_l1_end - lime_l1_start
ll2time = lime_l2_end - lime_l2_start
print(f'Speedup over Markup LIME: {lmtime / rtime}')
print(f'Speedup over Default LIME: {ldtime / rtime}')
print(f'Speedup LR LIME: {lltime / rtime}')
print(f'Speedup LR-l1 LIME: {ll1time / rtime}')
print(f'Speedup LR-l2 LIME: {ll1time / rtime}')


Speedup over Markup SHAP: 436.3959828877489
Speedup over Default SHAP: 272.85002832733113
Speedup LR SHAP: 121.77527101697238
Speedup LR-l1 SHAP: 120.86850007841231
Speedup LR-l2 SHAP: 120.86850007841231
Speedup over Markup LIME: 31.46806040432497
Speedup over Default LIME: 16.531128275480118
Speedup LR LIME: 3.1982016030019906
Speedup LR-l1 LIME: 2.778065603555822
Speedup LR-l2 LIME: 2.778065603555822


In [76]:
print(f'Time taken for RoT [init]: {rot_middle - rot_start}')

print(f'Time taken for Markup SHAP [init]: {shap_markup_middle - shap_markup_start}')
print(f'Time taken for Default SHAP [init]: {shap_def_middle - shap_def_start}')
print(f'Time taken for LR SHAP [init]: {shap_lr_middle - shap_lr_start}')
print(f'Time taken for LR-l1 SHAP [init]: {shap_l1_middle - shap_l1_start}')
print(f'Time taken for LR-l2 SHAP [init]: {shap_l2_middle - shap_l2_start}')

print(f'Time taken for Markup LIME [init]: {lime_markup_middle - lime_markup_start}')
print(f'Time taken for Default LIME [init]: {lime_def_middle - lime_def_start}')
print(f'Time taken for LR LIME [init]: {lime_lr_middle - lime_lr_start}')
print(f'Time taken for LR-l1 LIME [init]: {lime_l1_middle - lime_l1_start}')
print(f'Time taken for LR-l2 LIME [init]: {lime_l2_middle - lime_l2_start}')

Time taken for RoT [init]: 0.6732428073883057
Time taken for Markup SHAP [init]: 0.05002903938293457
Time taken for Default SHAP [init]: 0.014747142791748047
Time taken for LR SHAP [init]: 0.0010879039764404297
Time taken for LR-l1 SHAP [init]: 0.018313884735107422
Time taken for LR-l2 SHAP [init]: 0.004772186279296875
Time taken for Markup LIME [init]: 0.003010988235473633
Time taken for Default LIME [init]: 0.0031609535217285156
Time taken for LR LIME [init]: 0.005499124526977539
Time taken for LR-l1 LIME [init]: 0.005614757537841797
Time taken for LR-l2 LIME [init]: 0.0035169124603271484


- Time taken for RoT [init]: 0.6401269435882568
- Time taken for Markup SHAP [init]: 0.05836200714111328
- Time taken for Default SHAP [init]: 0.015105009078979492
- Time taken for LR SHAP [init]: 0.001422882080078125
- Time taken for LR-l1 SHAP [init]: 0.011024713516235352
- Time taken for LR-l2 SHAP [init]: 0.0037500858306884766
- Time taken for Markup LIME [init]: 0.0022079944610595703
- Time taken for Default LIME [init]: 0.008666038513183594
- Time taken for LR LIME [init]: 0.004256010055541992
- Time taken for LR-l1 LIME [init]: 0.0032219886779785156
- Time taken for LR-l2 LIME [init]: 0.0015611648559570312

In [77]:
print(f'Time taken for RoT: {rot_end - rot_start}')

print(f'Time taken for Markup SHAP: {shap_markup_end - shap_markup_start}')
print(f'Time taken for Default SHAP: {shap_def_end - shap_def_start}')
print(f'Time taken for LR SHAP: {shap_lr_end - shap_lr_start}')
print(f'Time taken for LR-l1 SHAP: {shap_l1_end - shap_l1_start}')
print(f'Time taken for LR-l2 SHAP: {shap_l2_end - shap_l2_start}')

print(f'Time taken for Markup LIME: {lime_markup_end - lime_markup_start}')
print(f'Time taken for Default LIME: {lime_def_end - lime_def_start}')
print(f'Time taken for LR LIME: {lime_lr_end - lime_lr_start}')
print(f'Time taken for LR-l1 LIME: {lime_l1_end - lime_l1_start}')
print(f'Time taken for LR-l2 LIME: {lime_l2_end - lime_l2_start}')

Time taken for RoT: 0.6750078201293945
Time taken for Markup SHAP: 294.57070112228394
Time taken for Default SHAP: 184.17590284347534
Time taken for LR SHAP: 82.19926023483276
Time taken for LR-l1 SHAP: 81.58718276023865
Time taken for LR-l2 SHAP: 81.50589799880981
Time taken for Markup LIME: 21.24118685722351
Time taken for Default LIME: 11.15864086151123
Time taken for LR LIME: 2.158811092376709
Time taken for LR-l1 LIME: 1.875216007232666
Time taken for LR-l2 LIME: 1.8898839950561523


- Time taken for RoT: 0.6402318477630615
- Time taken for Markup SHAP: 294.8998670578003
- Time taken for Default SHAP: 183.65712118148804
- Time taken for LR SHAP: 80.94095373153687
- Time taken for LR-l1 SHAP: 81.8488187789917
- Time taken for LR-l2 SHAP: 81.86487483978271
- Time taken for Markup LIME: 21.956696033477783
- Time taken for Default LIME: 11.080784797668457
- Time taken for LR LIME: 1.857715129852295
- Time taken for LR-l1 LIME: 1.88474702835083
- Time taken for LR-l2 LIME: 1.7988343238830566

In [78]:
rtime = rot_end - rot_start
smtime = shap_markup_end - shap_markup_start
sdtime = shap_def_end - shap_def_start
sltime = shap_lr_end - shap_lr_start
sl1time = shap_l1_end - shap_l1_start
sl2time = shap_l2_end - shap_l2_start

print(f'Speedup over Markup SHAP: {smtime / rtime}')
print(f'Speedup over Default SHAP: {sdtime / rtime}')
print(f'Speedup LR SHAP: {sltime / rtime}')
print(f'Speedup LR-l1 SHAP: {sl1time / rtime}')
print(f'Speedup LR-l2 SHAP: {sl1time / rtime}')


lmtime = lime_markup_end - lime_markup_start
ldtime = lime_def_end - lime_def_start
lltime = lime_lr_end - lime_lr_start
ll1time = lime_l1_end - lime_l1_start
ll2time = lime_l2_end - lime_l2_start
print(f'Speedup over Markup LIME: {lmtime / rtime}')
print(f'Speedup over Default LIME: {ldtime / rtime}')
print(f'Speedup LR LIME: {lltime / rtime}')
print(f'Speedup LR-l1 LIME: {ll1time / rtime}')
print(f'Speedup LR-l2 LIME: {ll1time / rtime}')


Speedup over Markup SHAP: 436.3959828877489
Speedup over Default SHAP: 272.85002832733113
Speedup LR SHAP: 121.77527101697238
Speedup LR-l1 SHAP: 120.86850007841231
Speedup LR-l2 SHAP: 120.86850007841231
Speedup over Markup LIME: 31.46806040432497
Speedup over Default LIME: 16.531128275480118
Speedup LR LIME: 3.1982016030019906
Speedup LR-l1 LIME: 2.778065603555822
Speedup LR-l2 LIME: 2.778065603555822


- Speedup over Markup SHAP: 460.6141792042459
- Speedup over Default SHAP: 286.86033321081567
- Speedup LR SHAP: 126.42444141812152
- Speedup LR-l1 SHAP: 127.84246685785381
- Speedup LR-l2 SHAP: 127.84246685785381
- Speedup over Markup LIME: 34.294913803793726
- Speedup over Default LIME: 17.30745641033662
- Speedup LR LIME: 2.901628740186949
- Speedup LR-l1 LIME: 2.943850786142619
- Speedup LR-l2 LIME: 2.943850786142619